In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Upload data
uploaded = files.upload()
data = pd.read_excel('resampled_loan_data.xlsx')

Saving resampled_loan_data.xlsx to resampled_loan_data (2).xlsx


In [ ]:
# Define target and variable of interest
target = 'loan_paid'
variable = 'emp_length'

# Ensure emp_length is treated as a categorical variable
data[variable] = data[variable].astype(str)  # Convert to string to handle any numeric formats
data[target] = pd.to_numeric(data[target], errors='coerce')  # Ensure target is numeric

# Handle missing values in emp_length (fill with mode or treat as separate category)
data[variable] = data[variable].fillna(data[variable].mode()[0])

# Calculate WoE
def calculate_woe(df, variable, target):
    # Create a contingency table
    woe_table = df.groupby(variable)[target].agg(['count', 'sum'])
    woe_table['good'] = woe_table['sum']  # Number of loan_paid = 1
    woe_table['bad'] = woe_table['count'] - woe_table['good']  # Number of loan_paid = 0

    # Total good and bad across all categories
    total_good = woe_table['good'].sum()
    total_bad = woe_table['bad'].sum()

    # Avoid division by zero by adding a small constant (0.5)
    woe_table['perc_good'] = (woe_table['good'] + 0.5) / (total_good + 0.5 * len(woe_table))
    woe_table['perc_bad'] = (woe_table['bad'] + 0.5) / (total_bad + 0.5 * len(woe_table))

    # Calculate WoE
    woe_table['woe'] = np.log(woe_table['perc_good'] / woe_table['perc_bad'])

    # Calculate Information Value (IV) for the variable
    woe_table['iv_contribution'] = (woe_table['perc_good'] - woe_table['perc_bad']) * woe_table['woe']
    iv = woe_table['iv_contribution'].sum()

    return woe_table[['good', 'bad', 'perc_good', 'perc_bad', 'woe']], iv

# Run WoE calculation
woe_table, iv = calculate_woe(data, variable, target)

# Display results
print(f"WoE Mapping for {variable}:")
print(woe_table)
print(f"\nInformation Value (IV) for {variable}: {iv:.4f}")
print("IV Interpretation: <0.02 (weak), 0.02-0.1 (medium), >0.1 (strong)")

# Create WoE mapping dictionary
woe_mapping = woe_table['woe'].to_dict()
print("\nWoE Mapping Dictionary:")
print(woe_mapping)

# Apply WoE transformation to the dataset
data[f'{variable}_woe'] = data[variable].map(woe_mapping)

# Verify transformation
print("\nSample of data with WoE transformation:")
print(data[[variable, f'{variable}_woe']].head())

WoE Mapping for emp_length:
             good    bad  perc_good  perc_bad       woe
emp_length                                             
1 year       6885   7021   0.065592  0.066847 -0.018950
10+ years   35166  32457   0.335002  0.309007  0.080772
2 years      9288   9411   0.088484  0.089601 -0.012546
3 years      8246   8474   0.078558  0.080680 -0.026663
4 years      6083   6221   0.057952  0.059231 -0.021821
5 years      6666   6381   0.063506  0.060754  0.044301
6 years      4934   4801   0.047007  0.045712  0.027932
7 years      4733   4606   0.045092  0.043856  0.027806
8 years      4877   4843   0.046464  0.046112  0.007605
9 years      4000   4035   0.038109  0.038419 -0.008101
< 1 year     8545   8609   0.081406  0.081966 -0.006852
nan          5545   8173   0.052827  0.077815 -0.387301

Information Value (IV) for emp_length: 0.0121
IV Interpretation: <0.02 (weak), 0.02-0.1 (medium), >0.1 (strong)

WoE Mapping Dictionary:
{'1 year': -0.018949630240604746, '10+ years': 0.0

In [ ]:
# Define target and variables (excluding pub_rec)
target = 'loan_paid'
numeric_vars = ['loan_amnt', 'annual_inc', 'dti', 'revol_util', 'total_bc_limit', 'tot_hi_cred_lim']
categorical_vars = ['term', 'emp_length', 'home_ownership', 'purpose', 'verification_status']
parent_vars_for_derived = ['installment']  # Needed to derive payment_to_inc
all_raw_vars = numeric_vars + categorical_vars + parent_vars_for_derived + [target]

# Subset data to relevant columns that exist in the raw dataset
data = data[all_raw_vars].copy()

# Handle missing values
for var in numeric_vars + parent_vars_for_derived:
    data[var] = pd.to_numeric(data[var], errors='coerce')
data[numeric_vars + parent_vars_for_derived] = data[numeric_vars + parent_vars_for_derived].fillna(
    data[numeric_vars + parent_vars_for_derived].median()
)
data[categorical_vars] = data[categorical_vars].fillna(data[categorical_vars].mode().iloc[0])

# Feature Engineering: Numeric Variables
# 1. loan_amnt: Box-Cox transformation
data['loan_amnt'], _ = stats.boxcox(data['loan_amnt'] + 1)  # Add 1 to handle zeros

# 2. annual_inc: Log transformation
data['annual_inc'] = np.log(data['annual_inc'] + 1)  # Add 1 to handle zeros

# 3. dti: Cap at 99th percentile
dti_99th = data['dti'].quantile(0.99)
data['dti'] = data['dti'].clip(upper=dti_99th)

# 4. revol_util: Cap at 100%
data['revol_util'] = data['revol_util'].clip(upper=100)

# 5. total_bc_limit: Box-Cox transformation
data['total_bc_limit'], _ = stats.boxcox(data['total_bc_limit'] + 1)  # Add 1 to handle zeros

# 6. tot_hi_cred_lim: Box-Cox transformation
data['tot_hi_cred_lim'], _ = stats.boxcox(data['tot_hi_cred_lim'] + 1)  # Add 1 to handle zeros

# 7. payment_to_inc: Derive and cap at 99th percentile
data['monthly_inc'] = data['annual_inc'] / 12
data['payment_to_inc'] = data['installment'] / data['monthly_inc']
pti_99th = data['payment_to_inc'].quantile(0.99)
data['payment_to_inc'] = data['payment_to_inc'].clip(upper=pti_99th)
data = data.drop(columns=['monthly_inc', 'installment'])  # Clean up temporary columns

# Feature Engineering: Categorical Variables
# 1. term: Dummy variable (1 for 60 months, 0 for 36 months)
data['term_60'] = (data['term'] == '60 months').astype(int)  # Adjust format if needed
data = data.drop(columns=['term'])

# 2. emp_length: Weight of Evidence (WoE) transformation
# Replace the placeholder WoE mapping with the calculated one
data['emp_length_woe'] = data['emp_length'].map(woe_mapping)
data = data.drop(columns=['emp_length'])  # Remove original column

# 3. home_ownership: Two dummy variables (RENT, MORTGAGE; OWN as reference)
data['home_ownership_RENT'] = (data['home_ownership'].isin(['RENT', 'ANY'])).astype(int)
data['home_ownership_MORTGAGE'] = (data['home_ownership'] == 'MORTGAGE').astype(int)
data = data.drop(columns=['home_ownership'])

# 4. purpose: Dummy variables for web app options, flagging high-risk purposes
high_risk_purposes = ['debt_consolidation', 'credit_card']  # Adjust as needed
data = pd.get_dummies(data, columns=['purpose'], prefix='purpose', drop_first=True)

# 5. verification_status: One-hot encode (drop one category)
data = pd.get_dummies(data, columns=['verification_status'], prefix='verification_status', drop_first=True)

# Update numeric variables list after transformations
numeric_vars_transformed = ['loan_amnt', 'annual_inc', 'dti', 'revol_util', 'total_bc_limit',
                           'tot_hi_cred_lim', 'payment_to_inc', 'emp_length_woe']

# Standardization: Apply z-scores to transformed numeric variables
scaler = StandardScaler()
data[numeric_vars_transformed] = scaler.fit_transform(data[numeric_vars_transformed])

# Final dataset check
print("Columns in preprocessed data:", data.columns.tolist())
print("Data types:\n", data.dtypes)
print("Sample of preprocessed data:\n", data.head())

# Save preprocessed data
data.to_csv('preprocessed_loan_data.csv', index=False)
print("Preprocessed data saved as 'preprocessed_loan_data.csv'")

# Download the CSV file to your computer
files.download('preprocessed_loan_data.csv')

Columns in preprocessed data: ['loan_amnt', 'annual_inc', 'dti', 'revol_util', 'total_bc_limit', 'tot_hi_cred_lim', 'loan_paid', 'payment_to_inc', 'term_60', 'emp_length_woe', 'home_ownership_RENT', 'home_ownership_MORTGAGE', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding', 'verification_status_Source Verified', 'verification_status_Verified']
Data types:
 loan_amnt                              float64
annual_inc                             float64
dti                                    float64
revol_util                             float64
total_bc_limit                         float64
tot_hi_cred_lim                        float64
loan_paid                                int64
payment_to_inc                         float64
term_60             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>